In [ ]:
!cp -r /kaggle/input/datasets/baohg153/prover9/Prover9-LADR-2026-4F /kaggle/working/

%cd /kaggle/working/Prover9-LADR-2026-4F
!make all STATIC=1
%cd /kaggle/working/

In [ ]:
!pip install pandas \
            seaborn \
            matplotlib \
            nltk \
            regex \
            torch \
            datasets \
            transformers \
            peft \
            nltk

In [ ]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
)
from peft import PeftModel

from nltk.sem.logic import LogicParser
from nltk.inference import Prover9
import regex as re  
import gc
from tqdm import tqdm
from torch.utils.data import DataLoader

import re
from collections import Counter

In [ ]:
class ManualDataset(torch.utils.data.Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        return {
            "id": item["id"],
            "premises": item["premises"],
            "conclusion": item["conclusion"],
            "label": item["label"]
        }

In [ ]:
class FOLModel:
    def __init__(
        self,
        model_name: str = "/kaggle/input/models/ductri0981/deepseek-model/transformers/default/1/deepseek_model",
        output_dir: str = "/kaggle/working/fol_model",
        max_length: int = 768,
        local_files_only=False
    ):        
        self.model_name = model_name
        self.output_dir = output_dir
        self.max_length = max_length
        
        #Load tokenizer
        print("Loading tokenizer...")
        self.tokenizer = AutoTokenizer.from_pretrained(
            model_name,
            trust_remote_code=True,
            local_files_only=local_files_only
        )
        #Load base model.
        print("Loading model...")
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            dtype=torch.float16,
            device_map={"": 0},
            trust_remote_code=True,
            local_files_only=True
        )
        self.tokenizer.pad_token = self.tokenizer.eos_token
        self.tokenizer.padding_side = "left"

        self.model.config.use_cache = False
        
    def load_finetune_model(self, path_adapter: str):
        # 🔥 Reset lại base model
        if hasattr(self, "model"):
            print("Cleaning old model...")
            del self.model
            torch.cuda.empty_cache()
            gc.collect()
        
        print("Reset base model")
        self.model = AutoModelForCausalLM.from_pretrained(
            self.model_name,
            dtype=torch.float16,
            device_map={"": 0},
            trust_remote_code=True,
            local_files_only=True
        )
        
        self.model = PeftModel.from_pretrained(
            self.model,
            path_adapter,
            local_files_only=True
        )
    
        print("Adapter loaded successfully")
    
        # ❗ KHÔNG merge
    
        # freeze weights (optional nhưng an toàn)
        for param in self.model.parameters():
            param.requires_grad = False
    
        self.model.config.use_cache = False
    
        torch.cuda.empty_cache()
        gc.collect()

    def build_prompt(self, premises, conclusion):
        sentence = premises + " " + conclusion
        formatted = [
            s.strip() for s in sentence.split('.') if s.strip()
        ]
        formatted = "\n".join(
            [f"{i+1}. {s}." for i, s in enumerate(formatted)]
        )
        
        prompt = f"""### Role
You are a precise Natural Language to First-Order Logic (FOL) translator. Your task is to perform a direct, line-by-line mapping of sentences into formal logic.

### Logical Symbols
Strictly use these symbols:
- Universal: ∀
- Existential: ∃
- Conjunction: ∧
- Disjunction: ∨
- Exclusive Or: ⊕
- Negation: ¬
- Implication: →
- Equivalence: ↔

### Strict Constraints
1. **One-to-One Parity**: The number of FOL expressions in the output must exactly match the number of sentences in the Natural Language input. 
2. **Line Separation**: Each FOL expression must be placed on a new line. Do not use any other delimiters or numbering.
3. **Predicate Consistency**: 
   - Use a single, consistent name for each property or relation. 
   - Do not use synonyms (e.g., if you use "Student(x)", do not use "is_student(x)" or "IsStudent(x)" elsewhere).
   - Prefer concise noun-based or verb-based naming (e.g., "Teacher(x)", "Loves(x, y)").
4. **No Metadata**: Output ONLY the FOL expressions. No explanations, no thought process, and no introductory remarks.

Now apply the rules strictly to the following input:

### Input:
{formatted}

### Output:
"""
        return prompt
        
    def predict(self, loader, max_new_tokens: int = 768, num_outputs = 10):
        self.model.eval()
        data = []
        with torch.no_grad():
            loop = tqdm(loader)
            for batch in loop:

                ids = batch["id"]
                premises = batch["premises"]
                conclusions = batch["conclusion"]
                labels = batch["label"]
                # ensure list
                if isinstance(premises, str):
                    premises = [premises]
                    conclusions = [conclusions]
                    ids = [ids]

                prompts = []
                for p, c in zip(premises, conclusions):
                    prompts.append(self.build_prompt(p, c))

                inputs = self.tokenizer(
                    prompts,
                    return_tensors="pt",
                    padding=True,
                    truncation=True,
                    max_length=self.max_length
                ).to(self.model.device)

                outputs = self.model.generate(
                    **inputs,
                    max_new_tokens=max_new_tokens,
                    do_sample=True,
                    temperature=1.0,
                    min_p=0.05,
    
                    num_return_sequences=num_outputs,
                    pad_token_id=self.tokenizer.eos_token_id
                )

                decoded_outputs = self.tokenizer.batch_decode(
                    outputs,
                    skip_special_tokens=True
                )
    
                batch_size = len(prompts)
    
                for i in range(batch_size):
    
                    sample_outputs = []
    
                    for j in range(num_outputs):
    
                        idx = i * num_outputs + j
                        text = decoded_outputs[idx]
    
                        if "### Output:" in text:
                            text = text.split("### Output:")[-1].strip()
                        else:
                            text = text[len(prompts[i]):].strip()
    
                        sample_outputs.append(text)
    
                    data.append({
                        "id": ids[i],
                        "natural": premises[i] + '\n' + conclusions[i],
                        "predictions": [
                            {"fol": fol, "label": None}
                            for fol in sample_outputs
                        ],
                        "label": labels[i]
                    })
        return data

In [ ]:
def lv_distance(s1, s2):
    if len(s1) < len(s2):
        return lv_distance(s2, s1)

    if len(s2) == 0:
        return len(s1)

    previous_row = range(len(s2) + 1)
    for i, c1 in enumerate(s1):
        current_row = [i + 1]
        for j, c2 in enumerate(s2):
            insertions = previous_row[j + 1] + 1
            deletions = current_row[j] + 1
            substitutions = previous_row[j] + (c1 != c2)
            current_row.append(min(insertions, deletions, substitutions))
        previous_row = current_row
    
    return previous_row[-1]

class DataFilter:
    def __init__(self):
        self.duplicate_count = 0
        self.syntax_error_count = 0
        self.length_ratio_count = 0
    
    def _is_valid_syntax(self, fol_str: str) -> bool:
        if not fol_str:
            return False
            
        stack = []
        matching_bracket = {')': '(', ']': '[', '}': '{'}
        
        for char in fol_str:                   
            if char in matching_bracket.values():
                stack.append(char)
            elif char in matching_bracket.keys():
                if not stack or stack.pop() != matching_bracket[char]:
                    return False

        return len(stack) == 0
    
    
    def _is_valid_length_ratio(self, nat_str: str, fol_str: str, lowerbound_ratio: float=0.5, upperbound_ratio: float=2.0) -> bool:
        nat_length = len(nat_str)
        if nat_length == 0:
            return False
    
        fol_length = len(fol_str)
        ratio = fol_length / nat_length
        
        return lowerbound_ratio <= ratio <= upperbound_ratio


    def _update_names(self, fol, ratio_threshold=1/3):
        pattern = r'(\w+)\('
        matches = re.findall(pattern, fol)
    
        counts = Counter(matches)
        all_names = list(set(matches))
    
        single_names = [name for name, count in counts.items() if count == 1]
    
        update_names = {}
    
        for n1 in single_names:
            for n2 in all_names:
                if n2 in single_names and len(n2) > len(n1):
                    continue
                if n1 != n2 and lv_distance(n1, n2) <= ratio_threshold * max(len(n1), len(n2)):
                    update_names[n1] = n2
    
        return update_names
    
    
    def filter(self, entry: dict, lowerbound_ratio:float=0.5, upperbound_ratio:float=2.0) -> dict:
        valid_predictions = []
        nat_str = entry.get("natural", "")
        
        natural_text = entry.get("natural", "").strip()
        sentences = [s for s in natural_text.split('.') if s.strip()]
        nat_length = len(sentences)
        
        for pred in entry.get("predictions", []):
            fol = pred.get("fol", "")
            
            # Validate syntax (bracket balancing)
            if not self._is_valid_syntax(fol):
                self.syntax_error_count += 1
                continue
            
            premise_list = [p.strip() for p in fol.split('\n') if p.strip()]
            conclusion = premise_list[-1]
            premise_list = list(set(premise_list[:-1]))

            # Check for length ratio
            if not self._is_valid_length_ratio(nat_str, fol, lowerbound_ratio, upperbound_ratio):
                self.length_ratio_count += 1
                continue

            valid_fol = "\n".join(premise_list + [conclusion])
            update = self._update_names(valid_fol)
            for key, value in update.items():
                pattern = fr'(?<![a-zA-Z0-9]){key}\('
                valid_fol = re.sub(pattern, value + "(", valid_fol)

            valid_predictions.append({'fol': valid_fol, 'label': None})
            
        filtered_element = entry.copy()
        filtered_element["predictions"] = valid_predictions

        return filtered_element

def filter2(fol_predictions, ground_truth, threshold: float = 0.5):
    if len(fol_predictions) == 0:
        return None

    count = {"True": 0, "False": 0, "Uncertain": 0}

    for prediction in fol_predictions:
        count[prediction["label"]] += 1

    if count[ground_truth] / len(fol_predictions) > threshold:
        return True
    return False

In [ ]:
class Engine:
    def __init__(self, prover9_path='C:\\Program Files (x86)\\Prover9-Mace4\\bin-win32'):
        self.parser = LogicParser()
        self.prover9_path = prover9_path
        self.prover = Prover9()
        self.prover.config_prover9(prover9_path)

    def _transform_xor(self, text):
        """
        Transform A ⊕ B to ¬(A ⟷ B)
        """
        while '⊕' in text:
            idx = text.find('⊕')
            
            # Find left component
            lhs_end = idx
            while lhs_end > 0 and text[lhs_end-1].isspace():
                lhs_end -= 1
                
            curr = lhs_end - 1
    
            if text[curr] == ')':
                balance = 0
                while curr >= 0:
                    if text[curr] == ')': balance += 1
                    elif text[curr] == '(': balance -= 1
                    curr -= 1
                    if balance == 0: break
                while curr >= 0 and (text[curr].isalnum() or text[curr] == '_'):
                    curr -= 1
                lhs_start = curr + 1
            else:
                while curr >= 0 and (text[curr].isalnum() or text[curr] == '_'):
                    curr -= 1
                lhs_start = curr + 1
                
            lhs = text[lhs_start:lhs_end]
    
            # Find right component
            rhs_start = idx + 1
            while rhs_start < len(text) and text[rhs_start].isspace():
                rhs_start += 1
                
            curr = rhs_start
            if text[curr].isalnum() or text[curr] == '_':
                while curr < len(text) and (text[curr].isalnum() or text[curr] == '_'):
                    curr += 1
                if curr < len(text) and text[curr] == '(':
                    balance = 0
                    while curr < len(text):
                        if text[curr] == '(': balance += 1
                        elif text[curr] == ')': balance -= 1
                        curr += 1
                        if balance == 0: break
                rhs_end = curr
            elif text[curr] == '(':
                balance = 0
                while curr < len(text):
                    if text[curr] == '(': balance += 1
                    elif text[curr] == ')': balance -= 1
                    curr += 1
                    if balance == 0: break
                rhs_end = curr
            else:
                rhs_end = curr + 1
                
            rhs = text[rhs_start:rhs_end]
    
            # Transform XOR
            new_expression = f"¬({lhs} ⟷ {rhs})"
            text = text[:lhs_start] + new_expression + text[rhs_end:]
    
        return text
    
    def _translate_fol(self, fol_text: str):
        # Transform XOR
        fol_text = self._transform_xor(fol_text)
        
        # '-' --> '_'
        fol_text = re.sub(r'(?<=[a-zA-Z0-9])-(?=[a-zA-Z0-9])', '_', fol_text)

        replacements = {
            '∀': 'all ', 
            '∃': 'exists ',
            '∧': '&', 
            '∨': '|',
            '⊕': '^',
            '¬': '-',
            '→': '->', 
            '⟷': '<->',
            '↔': '<->'
        }
        for k, v in replacements.items():
            fol_text = fol_text.replace(k, v)

        # Add dot: "all x (P(x))" --> "all x. (P(x))"
        fol_text = re.sub(r'(all|exists)\s+([a-z0-9]+)\s*', r'\1 \2. ', fol_text)

        # Fix prover9 constants name (eg: "yuri" -> "c_yuri")
        words = re.findall(r'\b[a-z][a-zA-Z0-9_]*\b', fol_text)
        reserved_words = {'all', 'exists', 'u', 'v', 'w', 'x', 'y', 'z'}
        
        for w in set(words):
            if w not in reserved_words:
                fol_text = re.sub(fr'\b{w}\b', f'c_{w}', fol_text)
        return fol_text


    def _is_valid_fol(self, fol_list):
        try:
            for line in fol_list:
                if line.strip():
                    self.parser.parse(line)
            return True
        except Exception:
            return False

    def _check_conclusion(self, premises, conclusion):
        '''
        This function is the engine. It checks whether the conclusion is True/False/Uncertain based on the list of premises.
        Args:
            premises: list[str]
            conclusion: str
        Returns: 
            str ("True"/"False"/"Uncertain")
        '''
        # Read fol strings
        translated_premises = [self._translate_fol(p) for p in premises]
        translated_conclusion = self._translate_fol(conclusion)

        if (not self._is_valid_fol(translated_premises) or not self._is_valid_fol([translated_conclusion])):
            error_msg = f"Invalid FOL syntax detected!"
            raise ValueError(error_msg)
        
        try:
            parsed_premises = [self.parser.parse(p) for p in translated_premises]
            parsed_conclusion = self.parser.parse(translated_conclusion)
        except Exception as e:
            raise f"Error: {e}"

        # Check conclusion
        if self.prover.prove(parsed_conclusion, []):
            raise Exception("Error: The conclusion itself is True!")
        elif self.prover.prove(parsed_conclusion.negate(), []):
            raise Exception("Error: The conclusion itself is False!")
        elif not self.prover.prove(None, parsed_premises):
            is_true = self.prover.prove(parsed_conclusion, parsed_premises)
            if is_true:
                return "True"

            is_false = self.prover.prove(parsed_conclusion.negate(), parsed_premises)
            if is_false:
                return "False"

            return "Uncertain"
        else: 
            raise Exception("Error: The premises are inconsistent!")
    
    def check_conclusion(self, data):
        fol_list = data["predictions"]
        new_fol_list = []
        error_list = []
        for fol in fol_list:
            try:
                premises = fol["fol"].split("\n")[:-1]
                conclusion = fol["fol"].split("\n")[-1]
                result = self._check_conclusion(premises, conclusion)

                if result == "True":
                    fol["label"] = "True"
                elif result == "False":   # FIX 2
                    fol["label"] = "False"
                else:
                    fol["label"] = "Uncertain"
                new_fol_list.append(fol)
            except Exception as e:
                error_list.append({
                    "premises": "\n".join(fol["fol"].split("\n")[:-1]),
                    "conclusion": fol["fol"].split("\n")[-1],
                    "error": repr(e)
                })
                continue
        data["predictions"] = new_fol_list
        return data, error_list

In [ ]:
def check_correct(fol_predictions, label):
    if len(fol_predictions) == 0:
        return None

    count = {"Uncertain": 0, "True": 0, "False": 0}

    for prediction in fol_predictions:
        count[prediction["label"]] += 1

    selected_answer = max(count, key=count.get)

    return selected_answer == label

In [ ]:
import json
with open("/kaggle/input/datasets/ductri0981/manual-test/manual_test.json", 'r', encoding="utf-8") as f:
    dataset = json.load(f)
dataset = ManualDataset(dataset)

loader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=False,
)

fol_model = FOLModel(
    model_name="/kaggle/input/models/ductri0981/deepseek-model/transformers/default/1/deepseek_model",
    local_files_only=True
)

In [ ]:
adapter_files = {
    "result_basic_2.json": "/kaggle/input/models/ductri0981/fol-model/transformers/default/2",
    "result_best_0-51.json": "/kaggle/input/models/ductri0981/best-0-51/transformers/default/1",
    "result_last_0-51.json": "/kaggle/input/models/ductri0981/last-0-51/transformers/default/1",
    "result_best_51-151.json": "/kaggle/input/models/ductri0981/best-51-151/transformers/default/1",
    "result_last_51-151.json": "/kaggle/input/models/ductri0981/last-51-151/transformers/default/1",
    "result_best_151-251.json": "/kaggle/input/models/ductri0981/best-151-251/transformers/default/1",
    "result_last_151-251.json": "/kaggle/input/models/ductri0981/last-151-251/transformers/default/1"
}
engine = Engine(prover9_path="/kaggle/working/Prover9-LADR-2026-4F/bin")
filter_1 = DataFilter()

In [ ]:
for result_file, path_adapter in adapter_files.items():
    fol_model.load_finetune_model(path_adapter)
    dataset_formated_predicted = fol_model.predict(loader)    
    dataset_fine_tune = []
    count_correct = 0
    for fol in dataset_formated_predicted:
        fol = filter_1.filter(fol)
        fol, error_list = engine.check_conclusion(fol)
        fol["error_data"] = error_list
        if check_correct(fol['predictions'], fol["label"]):
            count_correct += 1
        dataset_fine_tune.append(fol)

    accuracy = count_correct / len(dataset_fine_tune)
    results = {
        "accuracy": f"{accuracy:.4f}",
        "data": dataset_fine_tune
    }
    with open(result_file, 'w', encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=4)

    # 🔥 HARD CLEAN MEMORY
    del dataset_formated_predicted
    del dataset_fine_tune

    torch.cuda.empty_cache()
    gc.collect()